# PromptMaxxing

### Tashi - Rezi AI @tastengyal (on X)


## Motivation

- The prompt is a moat most AI application companies today.
- We all strive to make our AI inferences provide the best responses. In other words, the surrounding prompts are critical.
- However, maintaining and improving prompts iteratively while keeping it model agnostic is extremely slow and often un-scientific.
- If not done right, it results in ["Prompt Debt"](https://www.dbreunig.com/2026/06/22/the-problem-is-prompt-debt.html)

# A Solution We Found

### [DSPy](https://dspy.ai) - Program, don’t prompt your LLMs

- DSPy lets you declare a prompt as a typed **signature** and treat improving it as an *optimization problem* against an evaluation metric you trust. 
- Super fast iteration instead of hand-tweaking wording.
- With a clear evaluator, it turns prompt engineering into a measurable, repeatable loop.  

For any improvement (modify or add a new capability), you simply add/modify a criterion to the **evaluator**, not hand-edited instructions into your System Prompt.

# Example - Spell Backward

We'll walk through the loop end to end on a small, objectively checkable task:  

                spelling a word backward

Because there's exactly one right answer, the evaluator is plain exact matching — no LLM judge needed.

## The dataset

`spell_back.csv` holds `instruction, question, answer` rows — each `question` is a word and `answer` is that word reversed (e.g. `requiz → ziuqer`). The examples are drawn from the [Open Thoughts](https://www.open-thoughts.ai/) open reasoning dataset.

In [6]:
import dspy
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

# The LM we optimize.
lm = dspy.LM("openai/gpt-4o-mini", temperature=1, max_tokens=2048)
dspy.configure(lm=lm)

df = pd.read_csv("./spell_back.csv")
print(f"Loaded {len(df)} rows")
df.head()

Loaded 40 rows


,instruction,question,answer
0,Spell this word backward (example: sun -> nus):,requiz,ziuqer
1,Spell this word backward (example: sun -> nus):,getup,puteg
2,Spell this word backward (example: sun -> nus):,palpation,noitaplap
3,Spell this word backward (example: sun -> nus):,Karamojo,ojomaraK
4,Spell this word backward (example: sun -> nus):,cicala,alacic


## Step 1 — Baseline with DSPy

In [DSPy](https://dspy.ai) you declare a task as a typed **signature** — a spec of inputs and outputs whose docstring instruction becomes the actual prompt. This reframes prompt engineering as an *optimization problem*.

Our baseline is the *unoptimized* program: given a word, it returns the reversed word. That docstring instruction is exactly what the optimizer will later try to rewrite and beat.

In [12]:
class SpellBackward(dspy.Signature):
    """Spell the given word backward (example: sun -> nus)."""

    word: str = dspy.InputField(desc="The word to reverse.")
    reversed_word: str = dspy.OutputField(desc="The word spelled backward.")


baseline = dspy.Predict(SpellBackward)


def row_to_example(row) -> dspy.Example:
    return dspy.Example(word=row["question"], reversed_word=row["answer"]).with_inputs("word")


# 60/20/20 split with a seeded shuffle.
shuffled = df.sample(frac=1.0, random_state=0).reset_index(drop=True)
n = len(shuffled)
n_train, n_val = int(n * 0.6), int(n * 0.2)
trainset = [row_to_example(shuffled.iloc[i]) for i in range(n_train)]
valset = [row_to_example(shuffled.iloc[i]) for i in range(n_train, n_train + n_val)]
testset = [row_to_example(shuffled.iloc[i]) for i in range(n_train + n_val, n)]

print(f"{len(trainset)} train / {len(valset)} val / {len(testset)} test")

24 train / 8 val / 8 test


## Step 2 — The evaluator: exact match

To optimize anything, we need a metric to optimize towards. Because the task has exactly one correct answer, the evaluator is trivial and fixed: a prediction passes iff it equals the reversed word exactly (whitespace-trimmed).

We run the baseline over the held-out test split to get the number the optimizer has to beat.

In [13]:
from dspy.evaluate import Evaluate


def exact_match(gold, pred, trace=None) -> bool:
    got = (getattr(pred, "reversed_word", "") or "").strip()
    return got == gold.reversed_word.strip()


evaluator = Evaluate(devset=testset, metric=exact_match, num_threads=4, display_progress=True)
baseline_score = evaluator(baseline)
print(f"Baseline test accuracy: {baseline_score}")

Average Metric: 2.00 / 8 (25.0%): 100%|██████████| 8/8 [00:00<00:00, 1177.84it/s]

2026/07/02 17:55:43 INFO dspy.evaluate.evaluate: Average Metric: 2 / 8 (25.0%)



Baseline test accuracy: EvaluationResult(score=25.0, results=<list of 8 results>)


## Step 3 - Optimizing with GEPA

[GEPA](https://dspy.ai/api/optimizers/GEPA/overview/) (Genetic-Pareto) is DSPy's reflective prompt optimizer. It evolves the prompt's *instruction* in a loop:

1. **Run** the current prompt on a batch of training examples.
2. **Reflect** — a strong reflection model reads the metric's *textual feedback* (here, the correct answer for each miss) and proposes a rewritten instruction that targets those specific failures.
3. **Select** — candidates are scored on the validation split, and GEPA keeps a *Pareto frontier* of the best performers rather than a single winner, so it doesn't overfit to one kind of example.

In [14]:
def gepa_metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
    got = (getattr(pred, "reversed_word", "") or "").strip()
    expected = gold.reversed_word.strip()
    if got == expected:
        return dspy.Prediction(score=1.0, feedback="Correct.")
    feedback = f"Wrong. For '{gold.word}' the correct answer is '{expected}', but got '{got}'."
    return dspy.Prediction(score=0.0, feedback=feedback)


reflection_lm = dspy.LM("openai/gpt-5.4-mini", temperature=1.0, max_tokens=4096)

optimizer = dspy.GEPA(
    metric=gepa_metric,
    auto="light",
    reflection_lm=reflection_lm,
    reflection_minibatch_size=3,
    num_threads=4,
    track_stats=True,
    seed=0,
)

optimized = optimizer.compile(baseline, trainset=trainset, valset=valset)

2026/07/02 17:55:44 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 412 metric calls of the program. This amounts to 12.88 full evals on the train+val set.
2026/07/02 17:55:44 INFO dspy.teleprompt.gepa.gepa: Using 8 examples for tracking Pareto scores.
GEPA Optimization:   0%|          | 0/412 [00:00<?, ?rollouts/s]2026/07/02 17:55:44 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 8 (37.5%)
2026/07/02 17:55:44 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.375 over 8 / 8 examples
2026/07/02 17:55:44 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.375


Average Metric: 0.00 / 3 (0.0%): 100%|██████████| 3/3 [00:00<00:00, 1581.56it/s]

2026/07/02 17:55:44 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 3 (0.0%)
2026/07/02 17:55:44 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Proposed new text for self: You will be given a single input field:

- word: a string containing a word or name

Task:
- Reverse the order of the characters in the given word.
- Output only the reversed word, with no extra words, punctuation, or explanation.

Important:
- Preserve the exact characters as they appear, including uppercase/lowercase letters.
- Do not change spelling, add/remove letters, or normalize the word.
- Simply write the input word backward, character by character.

Examples:
- sun -> nus
- impoverish -> hsirevopmi
- Chachapuya -> ayupahcahC
- thrawcrook -> koorcwarht
2026/07/02 17:55:44 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/07/02 17:55:44 INFO dspy.teleprompt.gepa.gepa: Iteration 1: New subsample score 3.0 is better than old score 0.0. Continue to full eval and add to candidate pool.
2026/07/0


Average Metric: 2.00 / 3 (66.7%): 100%|██████████| 3/3 [00:00<00:00, 5368.14it/s] 

2026/07/02 17:55:44 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 3 (66.7%)
2026/07/02 17:55:44 INFO dspy.teleprompt.gepa.gepa: Iteration 2: Proposed new text for self: You will be given a single input field:

- word: a string containing a word or name

Task:
- Reverse the order of the characters in the given word exactly as written.
- Output only the reversed word, with no extra words, punctuation, labels, or explanation.

Important:
- Preserve the exact characters, including uppercase/lowercase letters and the original spelling.
- Do not add, remove, reorder, normalize, or otherwise modify any characters.
- Simply write the input word backward, character by character.

Examples:
- sun -> nus
- impoverish -> hsirevopmi
- Chachapuya -> ayupahcahC
- thrawcrook -> koorcwarht
- peltate -> etatlep
- sprint -> tnirps
- kecky -> ykcek
2026/07/02 17:55:44 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/07/02 17:55:44 INFO dspy.teleprompt.gepa.gepa: Iteration 2: New sub

...


Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:00<00:00, 1220.46it/s]

2026/07/02 17:55:44 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/07/02 17:55:44 INFO dspy.teleprompt.gepa.gepa: Iteration 35: All subsample scores perfect. Skipping.
2026/07/02 17:55:44 INFO dspy.teleprompt.gepa.gepa: Iteration 35: Reflective mutation did not propose a new candidate


GEPA Optimization: 100%|█████████▉| 411/412 [00:00<00:00, 526.64rollouts/s]2026/07/02 17:55:44 INFO dspy.teleprompt.gepa.gepa: Iteration 36: Selected program 17 score: 0.75


Average Metric: 1.00 / 3 (33.3%): 100%|██████████| 3/3 [00:00<00:00, 3821.11it/s]

2026/07/02 17:55:44 INFO dspy.evaluate.evaluate: Average Metric: 1.0 / 3 (33.3%)
2026/07/02 17:55:44 INFO dspy.teleprompt.gepa.gepa: Iteration 36: Proposed new text for self: You will be given a single input field:

- word: a string containing a word or name

Task:
- Reverse the exact sequence of characters in the given word.
- Output only the reversed word, with no extra words, punctuation, spaces, or explanation.

Important:
- Preserve every character exactly as it appears in the input, including uppercase/lowercase letters.
- Do not add, remove, or rearrange any characters except by reversing their order.
- Do not insert spaces or split the word.
- Do not normalize spelling or otherwise modify the input.
- The reversal must be exact and character-by-character; for example, if the input is "Hedera", the correct output is "aredeH" (not "Aredah" or any other altered form).

How to solve:
- Read the word from left to right.
- Write it back from right to left, character by character.

Ex

## Step 4 - Inspect the evolved instruction

GEPA optimizes the signature's *instruction* — compare it before vs. after.

In [15]:
print("=== BASELINE INSTRUCTION ===")
print(baseline.signature.instructions)
for name, predictor in optimized.named_predictors():
    print(f"\n=== OPTIMIZED: {name} ===")
    print(predictor.signature.instructions)

=== BASELINE INSTRUCTION ===
Spell the given word backward (example: sun -> nus).

=== OPTIMIZED: self ===
You will be given a single input field:

- word: a string containing a word or name

Task:
- Reverse the exact sequence of characters in the given word.
- Output only the reversed word, with no extra words, punctuation, spaces, or explanation.

Important:
- Preserve every character exactly as it appears in the input, including uppercase/lowercase letters.
- Do not add, remove, or rearrange any characters except by reversing their order.
- Do not insert spaces or split the word.
- Do not normalize spelling or otherwise modify the input.
- The reversal must be exact and character-by-character; for example, if the input is "Hedera", the correct output is "aredeH" (not "Aredah" or any other altered form).

How to solve:
- Read the word from left to right.
- Write it back from right to left, character by character.

Examples:
- sun -> nus
- impoverish -> hsirevopmi
- Chachapuya -> ayup

## Step 5 — Compare baseline vs optimized

Score both programs on the held-out test set (unseen during optimization), show every example side by side, and persist the winner.

In [16]:
def predict_all(program, examples):
    return [(getattr(program(word=ex.word), "reversed_word", "") or "").strip() for ex in examples]


expected = [ex.reversed_word.strip() for ex in testset]
base_out = predict_all(baseline, testset)
opt_out = predict_all(optimized, testset)

# Per-example side-by-side, with a ✅/❌ marker on each prediction.
compare = pd.DataFrame({
    "word": [ex.word for ex in testset],
    "expected": expected,
    "baseline": [f"{'✅' if b == e else '❌'} {b}" for b, e in zip(base_out, expected)],
    "optimized": [f"{'✅' if o == e else '❌'} {o}" for o, e in zip(opt_out, expected)],
})

n = len(testset)
base_hits = sum(b == e for b, e in zip(base_out, expected))
opt_hits = sum(o == e for o, e in zip(opt_out, expected))
base_acc, opt_acc = base_hits / n, opt_hits / n

line = "─" * 46
print(line)
print(f"{'RESULTS':^46}")
print(f"{f'held-out test set · {n} examples':^46}")
print(line)
print(f"  Baseline      {base_acc:6.1%}   ({base_hits}/{n})")
print(f"  Optimized     {opt_acc:6.1%}   ({opt_hits}/{n})")
print(f"  Improvement   {opt_acc - base_acc:+6.1%}")
print(line)

optimized.save("spell_back_gepa.json")
print("\n💾 Saved optimized program to spell_back_gepa.json")

compare

──────────────────────────────────────────────
                   RESULTS                    
        held-out test set · 8 examples        
──────────────────────────────────────────────
  Baseline       25.0%   (2/8)
  Optimized      37.5%   (3/8)
  Improvement   +12.5%
──────────────────────────────────────────────

💾 Saved optimized program to spell_back_gepa.json


,word,expected,baseline,optimized
0,serigraphy,yhpargires,❌ yhgarise res,❌ yhparigres
1,polyphage,egahpylop,✅ egahpylop,❌ egahpyloP
2,trisected,detcesirt,❌ detcisert,❌ detcisertet
3,fearlessly,ylsselraef,❌ ysselraef,❌ ylsraefle
4,convener,renevnoc,❌ revennoc,✅ renevnoc
5,jongleur,ruelgnoj,❌ regluonj,✅ ruelgnoj
6,Karamojo,ojomaraK,❌ ojomarak,❌ ojomarak
7,requiz,ziuqer,✅ ziuqer,✅ ziuqer


# Takeaways

We turned prompt engineering into a measurable, repeatable loop:

1. **Declare the task** as a DSPy signature — the docstring *is* the prompt.
2. **Define what "good" means** with a fixed evaluator. Here it's exact match; for open-ended tasks you'd swap in an LLM judge, but the loop is identical.
3. **Optimize automatically** — GEPA rewrites the instruction from the metric's written feedback, no hand-editing.

The mindset shift: when you have a new capability to add to an existing system prompt, don't hand-edit the prompt — add it to the eval criteria and let the LLMs optimize the prompt for you via optimizers like GEPA.

```mermaid
flowchart TD
    C["🆕 New capability to add"]

    C --> X["❌ Add it to the system prompt directly"]

    C --> E["✅ Add it to the eval criteria"]
    E --> G["Let LLMs optimize the prompt via GEPA"]
    G --> P["Updated system prompt"]

    classDef bad fill:#fdecea,stroke:#d9534f,color:#611a15;
    classDef good fill:#eafaf1,stroke:#28a745,color:#155724;
    class X,XR bad;
    class E,G,P,ER good;
```

# Putting human experts in the loop

The most valuable part is the evaluator/judge because it is what drives the loop. And that's where real field experience pays off: real lawyers, recruiters, clinicians — whoever owns the domain are better positioned to design and calibrate the evaluation criteria so the optimizer is chasing genuine quality, not a proxy that a developer assumes. They can also enrich the datasets by seeding or grading examples that make the synthetic dataset sharper. 

In a nutshell, bringing the net disribution of the llm inference system closer to real-world distribution still requires human experts.

### [Rezi](www.rezi.ai) is positioning itself to offer access to 4+ million experts across 100+ industries to help companies build and maintain state of the art AI Applications.

## References & further reading

- **DSPy** — signatures and optimizers: [dspy.ai](https://dspy.ai)
- **GEPA** — the optimizer behind Step 3. Agrawal et al., *"GEPA: Reflective Prompt Evolution Can Outperform Reinforcement Learning"* (2025), [arXiv:2507.19457](https://arxiv.org/abs/2507.19457)
- **Open Thoughts** — the open reasoning dataset the examples come from: [open-thoughts.ai](https://www.open-thoughts.ai/)